# NB_hospital_daily_summary

Builds the **gold.hospital_daily_summary** table — one row per hospital per calendar day with test volumes, rates, and day-over-day trends. Source: STTM Silver → Gold.

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_worksoace.gold.hospital_daily_summary
USING DELTA
AS
WITH daily_agg AS (
  SELECT
    lr.hospital_id,
    lr.test_date AS summary_date,
    COUNT(*) AS total_tests,
    COUNT(DISTINCT lr.patient_id) AS unique_patients_tested,
    COUNT(DISTINCT lr.doctor_id) AS active_doctor_count,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END) AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END) AS critical_test_count
  FROM databricks_worksoace.silver.lab_results lr
  GROUP BY lr.hospital_id, lr.test_date
),
with_rates AS (
  SELECT
    da.hospital_id,
    h.hospital_name,
    da.summary_date,
    c.day_name,
    c.is_weekend,
    da.total_tests,
    da.unique_patients_tested,
    da.active_doctor_count,
    da.abnormal_test_count,
    da.critical_test_count,
    CAST(da.abnormal_test_count / da.total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(da.critical_test_count / da.total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM daily_agg da
  LEFT JOIN databricks_worksoace.silver.hospitals h
    ON da.hospital_id = h.hospital_id
  LEFT JOIN databricks_worksoace.default.calender c
    ON c.date_key = CAST(date_format(da.summary_date, 'yyyyMMdd') AS INT)
)
SELECT
  hospital_id,
  hospital_name,
  summary_date,
  day_name,
  is_weekend,
  total_tests,
  unique_patients_tested,
  active_doctor_count,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests,
  CAST(
    CASE
      WHEN LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) IS NULL
        OR LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) = 0
      THEN NULL
      ELSE (total_tests - LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date))
        / CAST(LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS DOUBLE) * 100
    END AS DECIMAL(6,2)
  ) AS test_volume_change_pct
FROM with_rates;

In [0]:
record_count = spark.table("databricks_worksoace.gold.hospital_daily_summary").count()
dbutils.notebook.exit(str(record_count))